# STEP Files

Open every tracked STEP artifact in VS Code with the OCP CAD Viewer extension.

In [16]:
from __future__ import annotations

from dataclasses import dataclass
import json
from pathlib import Path
import subprocess
import sys

import build123d as bd
from ocp_vscode import show

_SUPPORTED_MODELED_ROLE = "tx_single_coil"
_EXPECTED_MODELED_BODY_NAMES = ("tx_pcb_l0", "tx_copper_l0")


@dataclass(frozen=True)
class StepArtifact:
    label: str
    step_path: Path
    generator_source: Path
    metadata_path: Path | None = None
    source_toml_path: Path | None = None
    generated: bool = False


def require_repo_root() -> Path:
    result = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    )
    root_text = result.stdout.strip()
    if not root_text:
        raise RuntimeError("git rev-parse --show-toplevel returned empty stdout")
    repo_root = Path(root_text).resolve()
    pyproject_path = repo_root / "pyproject.toml"
    if not pyproject_path.is_file():
        raise FileNotFoundError(f"repo root is missing pyproject.toml: {pyproject_path}")
    return repo_root


def require_file(repo_root: Path, repo_relative_path: Path) -> Path:
    absolute_path = repo_root / repo_relative_path
    if not absolute_path.is_file():
        raise FileNotFoundError(f"registered path is missing: {repo_relative_path}")
    return absolute_path


def _resolve_step_path(repo_root: Path, *, raw_step_path: str) -> Path:
    step_path = Path(raw_step_path)
    if not step_path.is_absolute():
        step_path = (repo_root / step_path).resolve()
    else:
        step_path = step_path.resolve()
    if not step_path.is_file():
        raise FileNotFoundError(f"modeled step path from metadata is missing: {step_path}")
    return step_path


def _discover_type2_generated_step(repo_root: Path, *, source_toml_path: Path) -> Path | None:
    ledger_roots = (
        repo_root / "run" / "step",
        repo_root / "examples" / "type2" / "artifacts",
    )
    candidates: list[Path] = []
    for ledger_root in ledger_roots:
        if not ledger_root.is_dir():
            continue
        for ledger_path in ledger_root.rglob("type2_step_ledger.json"):
            payload = json.loads(ledger_path.read_text(encoding="utf-8"))
            if not isinstance(payload, dict):
                continue
            raw_source_toml_path = payload.get("source_toml_path")
            if not isinstance(raw_source_toml_path, str) or raw_source_toml_path == "":
                continue
            if Path(raw_source_toml_path).resolve() != source_toml_path.resolve():
                continue
            raw_modeled_objects = payload.get("modeled_objects")
            if not isinstance(raw_modeled_objects, list):
                raise TypeError(f"type2_step_ledger modeled_objects must be a list: {ledger_path}")
            matching_objects = []
            for raw_modeled_object in raw_modeled_objects:
                if not isinstance(raw_modeled_object, dict):
                    raise TypeError(f"modeled object entry must be a table: {ledger_path}")
                raw_role = raw_modeled_object.get("role")
                if raw_role == _SUPPORTED_MODELED_ROLE:
                    matching_objects.append(raw_modeled_object)
            if len(matching_objects) == 0:
                continue
            if len(matching_objects) != 1:
                raise RuntimeError(
                    "type2 generated ledger must contain exactly one tx_single_coil modeled object "
                    f"(ledger={ledger_path}, count={len(matching_objects)})"
                )
            raw_step_path = matching_objects[0].get("step_path")
            if not isinstance(raw_step_path, str) or raw_step_path == "":
                raise TypeError(f"modeled object step_path must be non-empty string: {ledger_path}")
            raw_expected_names = matching_objects[0].get("expected_exported_body_names")
            if isinstance(raw_expected_names, str) or not isinstance(raw_expected_names, list):
                continue
            expected_names = tuple(raw_expected_names)
            if expected_names != _EXPECTED_MODELED_BODY_NAMES:
                continue
            candidates.append(_resolve_step_path(repo_root, raw_step_path=raw_step_path))
    unique_candidates = tuple(sorted(set(candidates), key=str))
    if not unique_candidates:
        return None
    if len(unique_candidates) != 1:
        raise RuntimeError(f"ambiguous type2 generated modeled STEP candidates: {unique_candidates}")
    return unique_candidates[0]


def ensure_generated_step(repo_root: Path, artifact: StepArtifact) -> Path:
    if not artifact.generated:
        return require_file(repo_root, artifact.step_path)
    require_file(repo_root, artifact.generator_source)
    subprocess.run(
        [
            sys.executable,
            str(repo_root / artifact.generator_source),
        ],
        cwd=repo_root,
        check=True,
    )
    if artifact.metadata_path is None:
        return require_file(repo_root, artifact.step_path)
    if artifact.source_toml_path is None:
        raise RuntimeError(f"generated artifact is missing source_toml_path: {artifact.label}")
    source_toml_path = require_file(repo_root, artifact.source_toml_path)
    discovered_step = _discover_type2_generated_step(repo_root, source_toml_path=source_toml_path)
    if discovered_step is None:
        raise RuntimeError(
            "type2 exporter produced no tx_single_coil generated STEP artifact for viewer"
        )
    expected_step_path = (repo_root / artifact.step_path).resolve()
    if discovered_step != expected_step_path:
        raise RuntimeError(
            "type2 generated STEP path mismatch "
            f"(expected={expected_step_path}, discovered={discovered_step})"
        )
    return discovered_step


REPO_ROOT = require_repo_root()

STEP_ARTIFACTS = (
    StepArtifact(
        label="type2 non-model scene",
        step_path=Path("run/step/type2/type2_non_model_scene.step"),
        source_toml_path=Path("examples/type2/type2.toml"),
        generator_source=Path("examples/type2/generate_non_model_step.py"),
        generated=True,
    ),
    StepArtifact(
        label="type2 tx_single_coil modeled object",
        step_path=Path("run/step/type2/objects/tx_rect_void_coil.step"),
        metadata_path=Path("run/step/type2/metadata/tx_rect_void_coil.metadata.json"),
        source_toml_path=Path("examples/type2/type2.toml"),
        generator_source=Path("examples/type2/generate_type2_step.py"),
        generated=True,
    ),
)

TRACKED_STEP_ARTIFACTS = tuple(artifact for artifact in STEP_ARTIFACTS if not artifact.generated)
GENERATED_STEP_ARTIFACTS = tuple(artifact for artifact in STEP_ARTIFACTS if artifact.generated)


In [17]:
tracked_result = subprocess.run(
    ["git", "ls-files", "*.step", "*.stp"],
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
TRACKED_STEP_PATHS = tuple(Path(line) for line in tracked_result.stdout.splitlines() if line)
REGISTERED_STEP_PATHS = tuple(artifact.step_path for artifact in STEP_ARTIFACTS)

unregistered_step_paths = tuple(sorted(set(TRACKED_STEP_PATHS) - set(REGISTERED_STEP_PATHS), key=str))
stale_registered_step_paths = tuple(
    sorted(
        set(artifact.step_path for artifact in TRACKED_STEP_ARTIFACTS) - set(TRACKED_STEP_PATHS),
        key=str,
    )
)
if unregistered_step_paths or stale_registered_step_paths:
    raise RuntimeError(
        "STEP notebook registry mismatch: "
        f"unregistered={unregistered_step_paths}, stale={stale_registered_step_paths}"
    )

for artifact in TRACKED_STEP_ARTIFACTS:
    require_file(REPO_ROOT, artifact.step_path)
    require_file(REPO_ROOT, artifact.generator_source)
for artifact in GENERATED_STEP_ARTIFACTS:
    require_file(REPO_ROOT, artifact.generator_source)
    if artifact.source_toml_path is None:
        raise RuntimeError(f"generated artifact is missing source_toml_path: {artifact.label}")
    require_file(REPO_ROOT, artifact.source_toml_path)

print("available STEP artifacts:")
for index, artifact in enumerate(STEP_ARTIFACTS):
    suffix = "generated" if artifact.generated else "tracked"
    print(f"{index}: {artifact.label} [{suffix}] -> {artifact.step_path}")


available STEP artifacts:
0: type2 non-model scene [generated] -> run/step/type2/type2_non_model_scene.step
1: type2 tx_single_coil modeled object [generated] -> run/step/type2/objects/tx_rect_void_coil.step


## Select and show STEP

Set `SELECTED_STEP_INDEX` to one of the printed indices above, then run this single viewer cell.

In [18]:
SELECTED_STEP_INDEX = 1

selected_artifact = STEP_ARTIFACTS[SELECTED_STEP_INDEX]
selected_step_path = ensure_generated_step(REPO_ROOT, selected_artifact)
selected_step = bd.import_step(selected_step_path)
shown_step = selected_step
shown_name = selected_artifact.label
if selected_artifact.label == "type2 tx_single_coil modeled object":
    copper_children = tuple(child for child in selected_step.children if child.label == "tx_copper_l0")
    if len(copper_children) != 1:
        raise RuntimeError(f"expected exactly one tx_copper_l0 child, found {len(copper_children)}")
    shown_step = copper_children[0]
    shown_name = "type2 tx_single_coil copper body"
show(
    shown_step,
    names=[shown_name],
)


source TOML: /home/harry/Projects/PythonProjects/peetsfea-main/examples/type2/type2.toml
output dir: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2
ledger JSON: /home/harry/Projects/PythonProjects/peetsfea-main/run/step/type2/type2_step_ledger.json
non-model object count: 7
modeled object count: 1
c
